# Глава Конфигурация

## Задача

**Валидация** - процесс проверки корректности данных. В вебе происходит всегда при отправке форм.

- Каждый тип валидации в таких системах обычно представлен классом-валидатором, который принимает на вход опции и предоставляет интерфейс в виде функции `validate`.
- Эта функция принимает на вход то что проверяется (валидируется) и возвращает словарь с ошибками.
- Если словарь пустой, значит ошибок нет.

### Реализуйте класс `PasswordValidator` ориентируясь на тесты.

- Этот валидатор поддерживает следующие опции:

  - `min_len` (по умолчанию 8) — минимальная длина пароля
  - `contain_numbers` (по умолчанию `False`) — требование содержать хотя бы одну цифру

- Метод `validate()`, который принимает пароль и возвращает словарь ошибок.
- Словарь ошибок в ключах содержит название опции, а в значении - текст указывающий на ошибку:

  - Если пароль слишком короткий, в словаре должна быть ошибка

  ```python
  {'min_len': 'too small'}
  ```

  - Если пароль должен содержать цифры, но их нет, в словаре ошибка

  ```python
  {'contain_numbers': 'should contain at least one number'}
  ```

  - Если ошибок нет, возвращается пустой словарь `{}`.

- При создании валидатора без параметров, применяются значения по умолчанию.
  - Если переданы параметры, то они должны переопределить стандартные.
  - Если же переданы неизвестные параметры, то валидатор должен их игнорировать.

### Подробнее в примере:

```python
validator = PasswordValidator()
errors = validator.validate('qwerty1')
print(errors)  # => {'min_len': 'too small'}

options = {'contain_numbers': True}
validator = PasswordValidator(**options)
errors = validator.validate('qwerty')
print(errors)  # => {'min_len': 'too small', 'contain_numbers': 'should contain at least one number'}

# валидатор должен игнорировать несуществующие опции
validator = PasswordValidator(numberz=None)
errors = validator.validate('qwertya3sdf')
print(errors) # => {}
```

### Подсказки

- Напомним, что обращаться к атрибутам класса можно через имя класса.
  - Например, к переменной `OPTIONS` можно обратиться как `PasswordValidator.OPTIONS`
- Обратите внимание, что конфигурация передается через keyword-параметры.
  - А значит что их можно упаковать в словарь, например как `def init(self, **kwargs)`

## Процедура решения задачи

1. Как организуется класс `PasswordValidator`

- `OPTIONS` — атрибут класса, а не экземпляра.
  - oн общий для всех объектов `PasswordValidator`.
  - Это типичный паттерн «конфигурация в одном месте»: если завтра минимальная длина пароля должна стать 10, ты меняешь значение один раз в словаре, а не ищешь по всему коду.

### Разбор `__init__`

```python
class PasswordValidator:
    OPTIONS = {
        'min_len': 8,
        'contain_numbers': False,
        }
    
    def __init__(self, **kwards):
       # Оставляем только те опции, которые есть в OPTIONS, остальные игнорируем
        self.options = {
            key: value 
            for key, value in kwards.items()
            if key in PasswordValidator.OPTIONS
        }
        # Для остальных используем значения по умолчанию из OPTIONS
        for key, default in PasswordValidator.OPTIONS.items():
            self.options.setdefault(key, default)
```

 #### <span style='color: red'>Что делает **kwargs</span>

Запись `def __init__(self, **kwargs)` означает: «прими любые именованные аргументы и положи их в словарь `kwargs`».

 #### **Пример вызова:**

```python
validator = PasswordValidator(min_len=10, contain_numbers=True, numberz=None)
```

Внутри `__init__` переменная `kwargs` будет такой:

```python
{
    'min_len': 10,
    'contain_numbers': True,
    'numberz': None
}
```

То есть `**kwargs` — это просто способ собрать все переданные именованные параметры в один словарь.

#### **Первое действие: фильтрация через генератор словаря**

``` python
self.options = {
    key: value
    for key, value in kwargs.items()
    if key in self.OPTIONS
}
```

Это генератор словаря *`dict comprehension`*. Он делает ровно то, что написано в комментарии:

- берёт пары «ключ–значение» из `kwargs`
- и оставляет только те, где ключ есть в `self.OPTIONS`.

**Если `self.OPTIONS` такой:**

```python
OPTIONS = {
    'min_len': 8,
    'contain_numbers': False,
}
```

то после этой строки self.options станет:

```python
{'min_len': 10, 'contain_numbers': True}
```

А `numberz` исчезнет — его ключ не содержится в `OPTIONS`, поэтому условие

```python
 `if key in self.OPTIONS`
```

 его отсекает. Вот это и есть «выбор»: мы явно решаем, какие ключи из переданных пользователем параметров считать валидными.

#### **Второе действие: подстановка значений по умолчанию**

```python
for key, default in PasswordValidator.OPTIONS.items():
            self.options.setdefault(key, default)
```

Здесь происходит следующее:

- Мы проходим по всем ключам и значениям из `OPTIONS` (это наши «дефолтные» настройки).
- Для каждого ключа вызываем `setdefault(key, default)`.
- `setdefault` работает так:
  - если ключ уже есть в словаре — ничего не делай;
  - если ключа нет — добавь его со значением default.

#### Зачем это нужно?

Чтобы даже если пользователь вообще ничего не передал:

```python
validator = PasswordValidator()
```

- у нас всё равно получился полный словарь опций на основе `OPTIONS`. 
- А если пользователь передал только часть опций — недостающие подтянутся из дефолтов.

#### Проверку на цифры делаем только если она включена

```python
if self.options.get('contain_numbers'):
    if not self._has_number(password):
        errors['contain_numbers'] = 'should contain at least one number'
```

##### Строка 1: `if self.options.get('contain_numbers')`

Здесь мы не делаем проверку на цифры, если она не включена. Это самое важное.

- `self.options` — это словарь с текущими настройками валидатора (собранный в `__init__` из переданных аргументов и значений по умолчанию).
- `'contain_numbers'` — ключ, который отвечает за требование «в пароле должна быть хотя бы одна цифра».
- `.get(key)` возвращает значение по ключу, а если ключа нет — вернёт `None`. В нашем случае ключ всегда есть (мы заполнили все опции в `__init__`), но `.get` — более безопасный стиль: он не выбросит KeyError, если вдруг логика изменится.

Так как по умолчанию `contain_numbers = False`, то при вызове `PasswordValidator()` условие будет `False` и блок внутри `if` просто не выполнится. То есть требование про цифры не применяется.
Почему это критично для тестов

##### Строка 2: `if not self._has_number(password)`

Эта строка выполняется только если требование про цифры включено (то есть только если первая проверка прошла).

- `self._has_number(password)` возвращает `True`, если в пароле есть хотя бы одна цифра, и `False`, если нет.
- `not` инвертирует результат:
  - если цифр нет → `_has_number` вернёт `False` → `not False` будет `True`→ заходим внутрь.
  - если цифры есть → `_has_number` вернёт `True` → `not True` будет `False` → внутрь не заходим.

То есть мы попадаем внутрь только тогда:

- когда требование есть,
- а цифры в пароле отсутствуют — это именно тот случай, когда нужно добавить ошибку.


##### Строка 3: `errors['contain_numbers'] = 'should contain at least one number'`

- `errors` — словарь, куда мы собираем все проблемы с паролем.
- Ключом служит имя правила `('contain_numbers')`, а значением — сообщение об ошибке.
- Если ошибок нет, словарь остаётся пустым